In [ ]:
from osm_graph import G

In [4]:
print(type(G))
print("Nodes:", len(G.nodes))
print("Edges:", len(G.edges))
print("Directed:", G.is_directed())
print("Multigraph:", G.is_multigraph())

print("Graph attributes:", G.graph)

<class 'networkx.classes.multidigraph.MultiDiGraph'>
Nodes: 49
Edges: 126
Directed: True
Multigraph: True
Graph attributes: {'created_date': '2026-10-20 16:05:35', 'created_with': 'OSMnx 2.1.1', 'crs': 'epsg:4326', 'simplified': True}


In [5]:
# Get the attributes of one node
node_id = next(iter(G.nodes))

print("Node ID:", node_id)
print("All attributes:", G.nodes[node_id])

# Access individual attributes
print("Longitude:", G.nodes[node_id].get("x"))
print("Latitude:", G.nodes[node_id].get("y"))

Node ID: 1496633981
All attributes: {'y': 27.722082, 'x': 85.2972932}
Longitude: 85.2972932
Latitude: 27.722082


In [6]:
u, v, key = next(iter(G.edges(keys=True)))

print("Source node:", u)
print("Destination node:", v)
print("Edge key:", key)

print("All attributes:")
print(G.edges[u, v, key])

print("Length:", G.edges[u, v, key].get("length"))
print("Road name:", G.edges[u, v, key].get("name"))
print("Road type:", G.edges[u, v, key].get("highway"))

Source node: 1496633981
Destination node: 1496675708
Edge key: 0
All attributes:
{'osmid': 136445396, 'highway': 'residential', 'oneway': False, 'reversed': True, 'length': np.float64(54.791914831444195)}
Length: 54.791914831444195
Road name: None
Road type: residential


In [2]:
print("Total nodes:", len(G.nodes))

for node_id, data in list(G.nodes(data=True))[:5]:
    print("Node ID:", node_id)
    print("Attributes:", data)
    print("---")

Total nodes: 49
Node ID: 1496633981
Attributes: {'y': 27.722082, 'x': 85.2972932}
---
Node ID: 1496634000
Attributes: {'y': 27.7156069, 'x': 85.2972868}
---
Node ID: 1496637546
Attributes: {'y': 27.7216453, 'x': 85.2980238}
---
Node ID: 1496637558
Attributes: {'y': 27.7199778, 'x': 85.2972671}
---
Node ID: 1496637578
Attributes: {'y': 27.7215562, 'x': 85.2979499}
---


In [3]:
print("Total directed edges:", len(G.edges))

for u, v, key, data in list(G.edges(keys=True, data=True))[:5]:
    print(f"From node: {u}")
    print(f"To node:   {v}")
    print(f"Edge key:  {key}")
    print(f"Attributes: {data}")
    print("---")

Total directed edges: 126
From node: 1496633981
To node:   1496675708
Edge key:  0
Attributes: {'osmid': 136445396, 'highway': 'residential', 'oneway': False, 'reversed': True, 'length': np.float64(54.791914831444195)}
---
From node: 1496633981
To node:   1496675782
Edge key:  0
Attributes: {'osmid': 136450522, 'highway': 'residential', 'name': 'Gojaga Marg', 'oneway': False, 'reversed': False, 'length': np.float64(102.74097246899649), 'geometry': <LINESTRING (85.297 27.722, 85.297 27.722, 85.296 27.722)>}
---
From node: 1496633981
To node:   1496637546
Edge key:  0
Attributes: {'osmid': 136450522, 'highway': 'residential', 'name': 'Gojaga Marg', 'oneway': False, 'reversed': True, 'length': np.float64(88.2026367474256), 'geometry': <LINESTRING (85.297 27.722, 85.298 27.722, 85.298 27.722, 85.298 27.722)>}
---
From node: 1496634000
To node:   1496675836
Edge key:  0
Attributes: {'osmid': 136445396, 'highway': 'residential', 'oneway': False, 'reversed': False, 'length': np.float64(577.16

In [ ]:

import math


def h(G, node, target):
    # Coordinates are stored as latitude (y) and longitude (x)
    lat1 = math.radians(G.nodes[node]["y"])
    lon1 = math.radians(G.nodes[node]["x"])

    lat2 = math.radians(G.nodes[target]["y"])  # convert to radians because python's trigo expects radians
    lon2 = math.radians(G.nodes[target]["x"])

    # Differences in radians
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Haversine formula
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    # Protect against floating-point rounding errors
    a = max(0.0, min(1.0, a))

    earth_radius_m = 6_371_000

    return 2 * earth_radius_m * math.asin(math.sqrt(a))


In [ ]:

import heapq


def initialize_search(start_node, end_node, G):
    forward_queue = [
        (h(G, start_node, end_node), start_node)
    ]

    backward_queue = [
        (h(G, end_node, start_node), end_node)
    ]

    forward_dist = {start_node: 0.0}
    backward_dist = {end_node: 0.0}

    forward_parent = {start_node: None}
    backward_parent = {end_node: None}

    return (
        forward_queue,
        backward_queue,
        forward_dist,
        backward_dist,
        forward_parent,
        backward_parent,
    )
